In [ ]:
!pip install google-genai pydantic pillow datasets -q

In [ ]:
# =====================================================================
# Top-level Step 1: Install and load all required packages
# =====================================================================

import os
import json
import random
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from datasets import load_dataset
from PIL import Image
from google.colab import userdata

# 🚀 Automatically retrieve the API key using Colab's built-in security mechanism
os.environ["GEMINI_API_KEY"] = userdata.get('gemini')
client = genai.Client()


# =====================================================================
# Top-level Step 2: Download the FUNSD dataset
# =====================================================================
print("⏳ Downloading the FUNSD dataset from Hugging Face...")
dataset = load_dataset("davidle7/funsd-json")
print("✅ Dataset loaded successfully!")

In [ ]:
# =====================================================================
# Top-level Step 3: Define data structures (Structured Outputs Schema)
# =====================================================================
class KeyValuePair(BaseModel):
    key: str = Field(description="Field name (question) in the form, e.g., Name, Date, Registration No.")
    value: str = Field(description="Specific content filled in the field (answer), e.g., John, 11/14/98, 533")

class ExtractedForm(BaseModel):
    pairs: list[KeyValuePair] = Field(description="List of all successfully paired key-value pairs in this form")

# =====================================================================
# Top-level Step 4: Define the Python Grader (Scoring Function)
# =====================================================================
def evaluate_predictions(worker_output: dict, ground_truth: dict) -> tuple[float, str]:
    """
    Compare the JSON generated by the Worker with the ground truth,
    calculate the matching accuracy, and generate an error log.
    """
    # 1. Parse the linking relationships in the official ground truth
    # FUNSD's native structure is complex; we simplify it into a set of (Key_Text, Value_Text) for comparison.
    gt_pairs = set()
    form_elements = ground_truth.get("form", [])

    # Create a mapping dictionary from ID to text
    id_to_text = {item["id"]: item["text"].strip().lower() for item in form_elements}
    id_to_label = {item["id"]: item["label"] for item in form_elements}

    # Establish correct relation pairs based on the official linking fields
    for item in form_elements:
        if "linking" in item and item["linking"]:
            for link in item["linking"]:
                id1, id2 = link[0], link[1]
                # Ensure we put the question first and the answer second
                if id_to_label.get(id1) == "question" and id_to_label.get(id2) == "answer":
                    gt_pairs.add((id_to_text.get(id1), id_to_text.get(id2)))
                elif id_to_label.get(id2) == "question" and id_to_label.get(id1) == "answer":
                    gt_pairs.add((id_to_text.get(id2), id_to_text.get(id1)))

    # 2. Parse the pairs extracted by the Worker model
    worker_pairs = set()
    for pair in worker_output.get("pairs", []):
        k = pair.get("key", "").strip().lower()
        v = pair.get("value", "").strip().lower()
        if k and v:
            worker_pairs.add((k, v))

    # 3. Calculate the score (intersection count / total official count)
    if not gt_pairs:
        return 100.0, "No available annotations for this form"

    correct_matches = worker_pairs.intersection(gt_pairs)
    missing_matches = gt_pairs - worker_pairs

    score = (len(correct_matches) / len(gt_pairs)) * 100

    # 4. Generate the error log for the Judge
    error_log = f"Current Score: {score:.1f}%\n"
    error_log += f"❌ Examples of missed or incorrectly recognized key fields:\n"
    for i, (missing_k, missing_v) in enumerate(list(missing_matches)[:3]):
        error_log += f"   - Should have captured: '{missing_k}' -> '{missing_v}', but the model missed it or paired it incorrectly.\n"

    return score, error_log

# =====================================================================
# Top-level Step 5: Main Program - Auto-Optimization Loop (Optimization Loop)
# =====================================================================

# A. Fixed on exam/sample number 23
sample_index = 23
sample_data = dataset['train'][sample_index]\

funsd_image = sample_data['image']
funsd_ground_truth = json.loads(sample_data['text_output'])

# B. Set the initial (1st generation) enhanced prompt
current_prompt = """You are a professional form OCR recognition expert. Please adhere to the following strict rules to extract Key-Value pairs:
1. [Standard Fields]: Pair the nearest and aligned question (Key) with its answer (Value).
2. [Table Data]: If you encounter a multi-column table, do not force it into a single Key-Value pair, and do not merge multiple rows of data into one giant Value string. Try to find the corresponding table header as the Key.
3. [Independent Stamps & Status]: Please look out for "receipt/received stamps (e.g., RECEIVED)" or "approval status (e.g., VIDEO TAPE APPROVAL)" at the edges of the image; these are usually independent Key-Value pairs as well.
4. Ensure the text matches the image perfectly; do not hallucinate layout or content."""

# History log: used to store the prompt of each generation
prompt_history = []
# 🌟 New: Prepare a JSON answer history log outside the loop
json_history = {}

print(f"\n🎯 Locking onto file #{sample_index} in the FUNSD training set for Prompt evolution!")
max_iterations = 3  # Set a maximum of 3 self-corrections

for iteration in range(1, max_iterations + 1):
    # At the beginning of each loop, back up this generation's Prompt into the history log
    prompt_history.append({"generation": iteration, "prompt": current_prompt})

    print(f"\n==================================================")
    print(f"🔄 【Generation {iteration} Prompt Iteration Started】")
    print(f"==================================================")
    print(f"💡 Current instruction (Prompt) used:\n\"{current_prompt}\"\n")

    # ─── Task A: Worker appears to take the test ───
    print("🤖 Calling Gemini 2.5 Flash for image recognition...")
    worker_response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=[funsd_image, current_prompt],
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=ExtractedForm, # Enforce structured output
            temperature=0.1 # Reduce randomness to ensure stability
        ),
    )

    try:
        worker_json = json.loads(worker_response.text)
    except Exception as e:
        print(f"💥 JSON parsing error: {e}")
        worker_json = {"pairs": []}

    # ─── Task B: Python teacher grades the answers ───
    score, error_report = evaluate_predictions(worker_json, funsd_ground_truth)
    print(f"\n📊 Grading Result: Received {score:.1f} points (Target: 85.0 points or above)")

    # 🌟 New: After successfully obtaining the score and JSON, save them immediately to the history log
    json_history[iteration] = {
        "score": score,
        "json_data": worker_json
    }

    # Check if passing score is achieved
    if score >= 85.0:
        print("🎉 Congratulations! The score has reached the target. Evolution complete! This is the final golden Prompt.")
        break

    if iteration == max_iterations:
        print("🛑 Maximum iterations reached. Evolution complete.")
        break

    # ─── Task C: Judge appears (Insufficent score, starting diagnosis and Prompt modification) ───
    print("\n🧠 Score below target, calling Gemini 2.5 Flash to optimize the prompt...")

    judge_system_instruction = """
    You are an AI Prompt Optimization Master. Your task is to analyze the reasons for errors made by the Worker model when extracting key-value pairs from paper forms, and to correct and reinforce the [Original Prompt].

    Please include specific "strategic guidance" regarding structure, spatial layout, alignment, or font treatment in the modified prompt (e.g., pay attention to horizontal alignment, pay attention to upper and lower adjacent cells).

    ⚠️ Constraints: Please directly output only the "optimized brand-new prompt content" itself. Do not include any introduction, quotation marks, or noise like "Sure, here is the modified prompt."
    """

    judge_user_content = f"""
    【Original Prompt】:
    "{current_prompt}"

    【Error Report and Missed Fields Provided by the Teacher】:
    {error_report}

    Please help me rewrite this prompt and add clear constraints so that the Worker won't make the same mistakes when encountering similar layouts again.
    """

    judge_response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=judge_user_content,
        config=types.GenerateContentConfig(
            system_instruction=judge_system_instruction,
            temperature=0.2  # 🌟 Slightly lower the temperature to make Flash Judge more calm and rational when modifying prompts
        )
    )

    # Overwrite Prompt, prepare to enter the next loop iteration
    current_prompt = judge_response.text.strip()

print(f"\n🏁 The final golden Prompt generated by evolution is:\n\"{current_prompt}\"")

# Neatly print out each generation's evolutionary process from the history log
print("\n📜 ==================================================")
print("📜           --- PROMPT EVOLUTION HISTORY LOG ---")
print("\n📜 ==================================================")
for history in prompt_history:
    print(f"【Generation {history['generation']} Prompt】:")
    print(f"\"{history['prompt']}\"")
    print("-" * 50)

In [ ]:
# =====================================================================
# View JSON answer comparison between Gen 1 and Gen 3
# =====================================================================

print("==================================================")
print(" 1️⃣【JSON Answer Generated by Gen 1 Prompt】")
print("==================================================")
if 1 in json_history:
    print(f"Score: {json_history[1]['score']:.1f}%")
    # indent=2 automatically formats and indents JSON to make it neat
    print(json.dumps(json_history[1]['json_data'], indent=2, ensure_ascii=False))
else:
    print("❌ No record for Generation 1")

print("\n\n==================================================")
print(" 3️⃣【JSON Answer Generated by Gen 3 Prompt】")
print("==================================================")
if 3 in json_history:
    print(f"Score: {json_history[3]['score']:.1f}%")
    print(json.dumps(json_history[3]['json_data'], indent=2, ensure_ascii=False))
elif 2 in json_history:
    # If the program reached the target in Generation 2 and ended early, show Gen 2 results
    print(f"Tip: The program reached the target in Generation 2 and ended early. Showing Gen 2 results below (Score: {json_history[2]['score']:.1f}%):")
    print(json.dumps(json_history[2]['json_data'], indent=2, ensure_ascii=False))
else:
    print("❌ No record for Generation 3 or target achievement")

In [ ]:
import json
from PIL import Image

# 1. Fetch file number 23
target_index = 23
exam_data = dataset['train'][target_index]

# 2. Obtain image and official ground truth
exam_image = exam_data['image']
exam_gt = json.loads(exam_data['text_output'])

# 3. Save the image to Colab's left folder
exam_image.save("/content/exam_23.jpg")
print("✅ Image successfully saved as 'exam_23.jpg'!")

# 4. Display this image directly inside Colab
print("\n👀 This is the image for exam/sample #23:")
display(exam_image)

# 5. (Optional) Print what the ground truth looks like for easy comparison
print("\n📜 Official Ground Truth (First 5 elements):")
for element in exam_gt.get("form", [])[:5]:
    print(f"ID {element['id']}: [{element['label']}] -> '{element['text']}'")